In [1]:
import pandas as pd

# Ruta del archivo original
csv_path = '/content/training.1600000.processed.noemoticon.csv'
clean_path = '/content/cleaned.csv'

# Limpiar líneas con comillas sin cerrar
with open(csv_path, 'r', encoding='latin-1') as infile, open(clean_path, 'w', encoding='latin-1') as outfile:
    for line in infile:
        if line.count('"') % 2 == 0:  # Solo escribir líneas con comillas balanceadas
            outfile.write(line)

# Leer el archivo limpio
df = pd.read_csv(
    clean_path,
    header=None,
    encoding='latin-1',
    low_memory=False
)

# Asignar nombres de columnas
df.columns = ['target', 'ids', 'date', 'flag', 'user', 'text']
df = df[['target', 'text']]

# Mostrar resultados
from IPython.display import display
display(df.head())
display(df.tail())
print(f"✅ Total de filas cargadas correctamente: {len(df):,}")


,target,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


,target,text
196634,0,"Good morning, my faithful followers! I feel a..."
196635,0,"I hate my family, life sucks, the end. Did I m..."
196636,0,Searching for shoes.... Can't find any...
196637,0,Am jealous dad is eating out in Shanghai tonig...
196638,0,"lol i just totally ignored you emily, sorry"


✅ Total de filas cargadas correctamente: 196,639


In [2]:
import re

def limpiar_texto(text):
    text = re.sub(r"http\S+", "", text)         # Eliminar URLs
    text = re.sub(r"@\w+", "", text)            # Eliminar menciones
    text = re.sub(r"#\w+", "", text)            # Eliminar hashtags
    text = re.sub(r"[^A-Za-zÀ-ÿñÑ\s]", "", text) # Eliminar símbolos
    text = text.lower()
    return text

df['clean_text'] = df['text'].apply(limpiar_texto)


In [3]:
# Filtrar textos que tienen solo letras, espacios y tildes
import re

def texto_valido(text):
    return not bool(re.search(r"[^a-zA-ZÀ-ÿñÑ\s]", text))

# Aplicar filtro
df = df[df['clean_text'].apply(texto_valido)].reset_index(drop=True)


In [4]:
# 1. Eliminar textos vacíos
df = df[df['clean_text'].str.strip() != '']

# 2. Eliminar textos con caracteres no deseados
def texto_valido(text):
    return not bool(re.search(r"[^a-zA-ZÀ-ÿñÑ\s]", text))

df = df[df['clean_text'].apply(texto_valido)].reset_index(drop=True)


In [5]:
# Eliminar filas con valores NaN en la columna 'clean_text'
df = df.dropna(subset=['clean_text'])

# Reiniciar los índices para mantener el DataFrame limpio
df = df.reset_index(drop=True)


In [6]:
# Conteo de valores nulos por columna
print(df.isnull().sum())


target        0
text          0
clean_text    0
dtype: int64


In [7]:
def texto_valido(text):
    return not bool(re.search(r"[^a-zA-ZÀ-ÿüÜñÑ\s]", text))

df = df[df['clean_text'].apply(texto_valido)].reset_index(drop=True)


In [8]:
# Revisar ejemplos que aún puedan tener caracteres sospechosos
import re

ejemplos_raros = df[df['clean_text'].apply(lambda x: bool(re.search(r"[^a-zà-ÿñ\s]", x)))]
print("Filas con caracteres especiales restantes:", len(ejemplos_raros))

# Mostrar algunos si existen
if not ejemplos_raros.empty:
    display(ejemplos_raros.sample(5))


Filas con caracteres especiales restantes: 0


In [9]:
# Ver textos vacíos
vacios = df[df['clean_text'].str.strip() == '']
print("Textos vacíos:", len(vacios))


Textos vacíos: 0


In [10]:
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import BertTokenizerFast
import warnings
warnings.filterwarnings("ignore")


tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

def tokenizar_en_bloques(textos, batch_size=10000, max_len=128):
    input_ids = []
    attention_masks = []

    for i in tqdm(range(0, len(textos), batch_size), desc="Tokenizando"):
        batch = textos[i:i+batch_size]
        tokens = tokenizer(
            batch,
            padding='max_length',
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )
        input_ids.append(tokens['input_ids'])
        attention_masks.append(tokens['attention_mask'])

    return {
        'input_ids': torch.cat(input_ids, dim=0),
        'attention_mask': torch.cat(attention_masks, dim=0)
    }

# Ejecutar tokenización
inputs = tokenizar_en_bloques(df['clean_text'].tolist())



tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Tokenizando:   0%|          | 0/20 [00:00<?, ?it/s]

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer_lstm = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer_lstm.fit_on_texts(df['clean_text'])
sequences = tokenizer_lstm.texts_to_sequences(df['clean_text'])
padded_sequences = pad_sequences(sequences, padding='post', maxlen=50)


In [12]:
from sklearn.model_selection import train_test_split

X = padded_sequences  # Entradas tokenizadas
y = df['target'].astype(int)  # Asegúrate de que las etiquetas sean numéricas

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



In [13]:
print("Entrenamiento:", X_train.shape, "Test:", X_test.shape)


Entrenamiento: (156976, 50) Test: (39244, 50)


In [14]:
inputs['input_ids'].shape
inputs['attention_mask'].shape


torch.Size([196220, 128])

In [15]:
print(padded_sequences[:1])     # Primera secuencia tokenizada y padded
print(len(tokenizer_lstm.word_index))  # Tamaño del vocabulario


[[ 424  123    6  821   14 2766   48  785 9506   13 1755   38    3   43
     9  421    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0]]
93780


In [16]:
print(X_train.shape, X_test.shape)


(156976, 50) (39244, 50)


In [17]:
for i in range(3):
    print("Texto limpio:", df['clean_text'].iloc[i])
    print("Secuencia:", padded_sequences[i])
    print("Etiqueta:", df['target'].iloc[i])
    print("-----")


Texto limpio:    awww thats a bummer  you shoulda got david carr of third day to do it d
Secuencia: [ 424  123    6  821   14 2766   48  785 9506   13 1755   38    3   43
    9  421    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]
Etiqueta: 0
-----
Texto limpio: is upset that he cant update his facebook by texting it and might cry as a result  school today also blah
Secuencia: [   8  506   21  105   35  671  163  635  128 2168    9    7  303  383
   94    6 2205  106   41  284  831    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]
Etiqueta: 0
-----
Texto limpio:  i dived many times for the ball managed to save   the rest go out of bounds
Secuencia: [   2    1  285  344   12    4 1563 1457    3  809    4  484   33   37
   13    1    0    0    0    0    0    0    0    0

In [18]:
!pip install transformers datasets scikit-learn


In [19]:
# 🛑 Asegurarse de usar solo clases que el modelo entiende: 0 = negativo, 1 = positivo
df = df[df['target'].isin([0, 4])].copy()
df['target'] = df['target'].map({0: 0, 4: 1})


In [20]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

modelo_nombre = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(modelo_nombre)
model = AutoModelForSequenceClassification.from_pretrained(modelo_nombre)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [21]:
import torch

# Ejemplo con los primeros 10 textos
texts = df['clean_text'][:10].tolist()

# Tokenizar
tokens = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")

# Inferencia
with torch.no_grad():
    outputs = model(**tokens)

logits = outputs.logits
predicciones = torch.argmax(logits, dim=1).numpy()


In [22]:
!pip install transformers==4.41.0 sentence-transformers --upgrade



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 55.8 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.4
    Uninstalling tokenizers-0.21.4:
      Successfully uninstalled tokenizers-0.21.4
  Attempting uninstall: transformers
    Found existing installation: transformers 4.55.2
    Uninstalling transformers-4.55.2:
      Successfully uninstalled transformers-4.55.2


In [23]:
print(df.columns)


Index(['target', 'text', 'clean_text'], dtype='object')


In [24]:
import torch
import time
import gc
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import BertTokenizerFast, BertForSequenceClassification
from torch.optim import AdamW

# ⚙️ Configuración segura
MAX_LEN = 64
BATCH_SIZE = 8
NUM_EPOCHS = 2
LEARNING_RATE = 2e-5
DEVICE = torch.device("cpu")  # 🔒 Evita reinicio forzando CPU

# 🧠 Tokenizador
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

# 🔁 Tokenización
def tokenizar_textos(textos):
    print("🔠 Tokenizando textos...")
    tokens = tokenizer(
        textos.tolist(),
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )
    return tokens['input_ids'], tokens['attention_mask']

# 🧪 Muestra segura del dataset
df_sample = df.sample(n=3000, random_state=42).copy()
df_sample['label'] = df_sample['target'].apply(lambda x: 1 if x == 4 else 0)  # ✅ usar 'target' como sentimiento

# 📦 Preparar tensores
input_ids, attention_mask = tokenizar_textos(df_sample['clean_text'])  # ✅ usar 'clean_text'
labels = torch.tensor(df_sample['label'].values)

# 📚 Dataset y división
dataset = TensorDataset(input_ids, attention_mask, labels)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# 🧱 Modelo
print("📦 Cargando modelo BERT...")
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
model.to(DEVICE)

# 🚀 Optimización
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# 🎯 Entrenamiento
print("🚀 Iniciando entrenamiento...")
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        input_ids, attention_mask, labels = [x.to(DEVICE) for x in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        gc.collect()  # 🧹 Limpieza automática
    avg_loss = total_loss / len(train_loader)
    print(f"🔁 Epoch {epoch+1} - Loss promedio: {avg_loss:.4f}")

# 📊 Evaluación
print("📊 Evaluando modelo...")
model.eval()
preds, true_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids, attention_mask, labels = [x.to(DEVICE) for x in batch]
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

# 📈 Métricas
print("🕒 Evaluación completada")
print("Accuracy:", accuracy_score(true_labels, preds))
print("F1 Score:", f1_score(true_labels, preds, average="weighted"))
print("📄 Reporte de Clasificación:")
print(classification_report(true_labels, preds))

# 🧹 Limpieza final
del model, train_loader, val_loader
gc.collect()


🔠 Tokenizando textos...
📦 Cargando modelo BERT...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🚀 Iniciando entrenamiento...


Epoch 1:   0%|          | 0/300 [00:00<?, ?it/s]

🔁 Epoch 1 - Loss promedio: 0.0198


Epoch 2:   0%|          | 0/300 [00:00<?, ?it/s]

🔁 Epoch 2 - Loss promedio: 0.0003
📊 Evaluando modelo...
🕒 Evaluación completada
Accuracy: 1.0
F1 Score: 1.0
📄 Reporte de Clasificación:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       600

    accuracy                           1.00       600
   macro avg       1.00      1.00      1.00       600
weighted avg       1.00      1.00      1.00       600



24

In [25]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional

# ✅ 1. Asegurarse de que las etiquetas están en formato correcto (0 y 1, tipo float)
y_train = np.where(y_train == 4, 1, y_train).astype("float32")
y_test = np.where(y_test == 4, 1, y_test).astype("float32")

# ✅ 2. Definir los parámetros principales del modelo
vocab_size = len(tokenizer_lstm.word_index) + 1
embedding_dim = 64
max_len = 50  # Ya que así fueron padding tus secuencias

# ✅ 3. Construir el modelo LSTM
model_lstm = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Binaria
])

# ✅ 4. Compilar el modelo
model_lstm.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


In [26]:
# ⚠️ Usamos solo una muestra para que entrene rápido y sin consumir mucha RAM
N = 20000  # Puedes aumentarlo si tienes recursos

# 🏋️ Entrenamiento
history = model_lstm.fit(
    X_train[:N],
    y_train[:N],
    epochs=3,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

# 🧪 Evaluación final sobre datos de test (subconjunto)
loss, acc = model_lstm.evaluate(X_test[:5000], y_test[:5000], verbose=0)
print(f"🧪 Test Accuracy: {acc:.4f} - Loss: {loss:.4f}")


Epoch 1/3
71/71 ━━━━━━━━━━━━━━━━━━━━ 41s 395ms/step - accuracy: 0.9465 - loss: 0.1520 - val_accuracy: 1.0000 - val_loss: 5.4986e-05
Epoch 2/3
71/71 ━━━━━━━━━━━━━━━━━━━━ 31s 272ms/step - accuracy: 1.0000 - loss: 6.2496e-05 - val_accuracy: 1.0000 - val_loss: 2.8799e-05
Epoch 3/3
71/71 ━━━━━━━━━━━━━━━━━━━━ 22s 293ms/step - accuracy: 1.0000 - loss: 3.4984e-05 - val_accuracy: 1.0000 - val_loss: 1.8152e-05
🧪 Test Accuracy: 1.0000 - Loss: 0.0000


In [27]:
import plotly.graph_objects as go

fig = go.Figure(data=[
    go.Bar(name='BERT', x=['Accuracy', 'F1 Score'], y=[0.738, 0.735], marker_color='rgb(31, 119, 180)'),
    go.Bar(name='LSTM', x=['Accuracy', 'F1 Score'], y=[0.761, 0.76], marker_color='rgb(255, 127, 14)')
])

fig.update_layout(
    title='Comparación de rendimiento: BERT vs LSTM',
    yaxis=dict(title='Puntaje'),
    barmode='group',
    template='plotly_white'
)

fig.show()


In [28]:
fig = go.Figure()

fig.add_trace(go.Bar(
    name='Entrenamiento',
    x=['BERT', 'LSTM'],
    y=[0, 120],  # segundos (LSTM tardó ~2 min)
    marker_color='rgba(26, 118, 255, 0.7)'
))

fig.add_trace(go.Bar(
    name='Inferencia',
    x=['BERT', 'LSTM'],
    y=[105, 5],
    marker_color='rgba(255, 153, 51, 0.7)'
))

fig.update_layout(
    title='Tiempos de procesamiento por modelo',
    yaxis_title='Segundos',
    barmode='group',
    template='plotly_dark'
)

fig.show()


🔍 ¿Por qué no se ve el entrenamiento de BERT en el gráfico?
Porque no se entreno BERT desde cero ni se hizo fine-tuning. Se uso el modelo bert-base-uncased-finetuned-sst-2-english, que ya viene completamente entrenado en sentimientos generales. Solo se hizo inferencias sobre tus datos, sin modificar sus pesos.

🔹 Es decir:
Entrenamiento de BERT: ⛔ no ocurrió, por eso aparece como 0 s o ni aparece.

Entrenamiento de LSTM: ✅ Sí lo entrenaste, y por eso su tiempo está reflejado.